In [1]:
import numpy as np
import pandas as pd

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from utils.metric import metric
import joblib

np.random.seed(42)
pd.set_option("display.max_columns", 100)

In [2]:
HORIZONS = [12, 24, 48, 72]

bundle = joblib.load("../models/stack_bundle.joblib")

W = bundle["W"]
w_risk = bundle["w_risk"]
best_p = bundle["best_p"]
post_isos = bundle["post_isos"]

In [3]:
train = pd.read_csv("../../data/train.csv")
val   = pd.read_csv("../../data/val.csv")

print("Train shape:", train.shape)
print("Val shape:", val.shape)

Train shape: (198, 37)
Val shape: (23, 37)


In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    eps = 1e-6

    dist = d["dist_min_ci_0_5h"].astype(float).clip(lower=1.0)  # meters
    dist_km = dist / 1000.0

    perims = d["num_perimeters_0_5h"].astype(float).clip(lower=0.0)
    dt_span = d["dt_first_last_0_5h"].astype(float).clip(lower=0.0)

    closing = d["closing_speed_m_per_h"].astype(float)
    closing_pos = closing.clip(lower=0.0)

    radial_rate = d["radial_growth_rate_m_per_h"].astype(float).clip(lower=0.0)
    along_track = d["along_track_speed"].astype(float) if "along_track_speed" in d.columns else 0.0

    align_abs = d["alignment_abs"].astype(float).clip(lower=0.0, upper=1.0)
    align_cos = d["alignment_cos"].astype(float) if "alignment_cos" in d.columns else 0.0

    area = d["area_first_ha"].astype(float).clip(lower=0.0)
    growth_rate_ha_h = d["area_growth_rate_ha_per_h"].astype(float).clip(lower=0.0)

    d["dist_km"] = dist_km
    d["log_dist"] = np.log1p(dist)
    d["sqrt_dist"] = np.sqrt(dist)

    d["inv_dist"] = 1.0 / (dist + 1.0)
    d["inv_dist_sq"] = d["inv_dist"] ** 2
    d["inv_dist_km"] = 1.0 / (dist_km + 0.05)

    d["zone_lt3km"] = (dist < 3000).astype(int)
    d["zone_3to5km"] = ((dist >= 3000) & (dist < 5000)).astype(int)
    d["zone_5to10km"] = ((dist >= 5000) & (dist < 10000)).astype(int)
    d["zone_ge10km"] = (dist >= 10000).astype(int)

    if "dist_fit_r2_0_5h" in d.columns:
        r2 = d["dist_fit_r2_0_5h"].astype(float).clip(lower=0.0, upper=1.0)
        d["dist_trend_reliable"] = (r2 > 0.6).astype(int)
        d["dist_r2"] = r2
    else:
        d["dist_trend_reliable"] = 0
        d["dist_r2"] = 0.0

    for col in ["dist_std_ci_0_5h", "dist_change_ci_0_5h", "dist_slope_ci_0_5h", "dist_accel_m_per_h2"]:
        if col in d.columns:
            d[col] = d[col].astype(float)

    if "dist_change_ci_0_5h" in d.columns:
        d["dist_change_km"] = d["dist_change_ci_0_5h"] / 1000.0
        d["dist_change_norm"] = d["dist_change_ci_0_5h"] / (dist + 1.0)

    if "dist_slope_ci_0_5h" in d.columns:
        d["dist_slope_norm"] = d["dist_slope_ci_0_5h"] / (dist + 1.0)

    if "dist_std_ci_0_5h" in d.columns:
        d["dist_std_norm"] = d["dist_std_ci_0_5h"] / (dist + 1.0)

    effective_closing = closing_pos + radial_rate
    d["effective_closing_speed"] = effective_closing

    d["eta_closing_h"] = np.where(closing_pos > 0.1, dist / (closing_pos + eps), 9999.0)
    d["eta_effective_h"] = np.where(effective_closing > 0.1, dist / (effective_closing + eps), 9999.0)

    d["log_eta_effective"] = np.log1p(np.clip(d["eta_effective_h"], 0, 9999))
    d["log_eta_closing"] = np.log1p(np.clip(d["eta_closing_h"], 0, 9999))

    for h in HORIZONS:
        d[f"eta_within_{h}h"] = (d["eta_effective_h"] <= float(h)).astype(int)
        d[f"eta_margin_{h}h"] = d["eta_effective_h"] - float(h)

    if isinstance(along_track, pd.Series):
        along_pos = along_track.astype(float).clip(lower=0.0)
        d["eta_alongtrack_h"] = np.where(along_pos > 0.1, dist / (along_pos + eps), 9999.0)
        d["log_eta_alongtrack"] = np.log1p(np.clip(d["eta_alongtrack_h"], 0, 9999))
    else:
        d["eta_alongtrack_h"] = 9999.0
        d["log_eta_alongtrack"] = np.log1p(9999.0)

    fire_radius_m = np.sqrt((area * 10000.0) / np.pi)
    d["fire_radius_m"] = fire_radius_m
    d["radius_to_dist"] = fire_radius_m / (dist + 1.0)

    d["dist_minus_radius"] = np.clip(dist - fire_radius_m, 1.0, None)
    d["log_dist_minus_radius"] = np.log1p(d["dist_minus_radius"])

    d["area_to_dist_km"] = area / (dist_km + 0.1)
    d["growth_to_dist_km"] = growth_rate_ha_h / (dist_km + 0.1)

    if "radial_growth_m" in d.columns:
        d["radial_growth_m"] = d["radial_growth_m"].astype(float).clip(lower=0.0)
        d["radial_to_dist"] = d["radial_growth_m"] / (dist + 1.0)

    d["threat_pressure"] = align_abs * effective_closing / (np.log1p(dist) + eps)
    d["alignment_x_closing"] = align_abs * closing_pos
    d["alignment_x_effective"] = align_abs * effective_closing

    if "cross_track_component" in d.columns:
        ctc = d["cross_track_component"].astype(float)
        d["cross_track_abs"] = np.abs(ctc)
        d["cross_track_norm"] = np.abs(ctc) / (dist + 1.0)

    if "spread_bearing_sin" in d.columns and "spread_bearing_cos" in d.columns:
        sb_sin = d["spread_bearing_sin"].astype(float)
        sb_cos = d["spread_bearing_cos"].astype(float)
        d["bearing_strength"] = np.sqrt(sb_sin**2 + sb_cos**2) 

    d["has_movement"] = (perims > 1).astype(int)
    d["perim_density"] = perims / (dt_span + 0.25) 
    d["short_window"] = (dt_span < 0.5).astype(int)

    if "low_temporal_resolution_0_5h" in d.columns:
        d["low_temporal_resolution_0_5h"] = d["low_temporal_resolution_0_5h"].astype(int)

    hour = d["event_start_hour"].astype(float) if "event_start_hour" in d.columns else 0.0
    d["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    d["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)

    dow = d["event_start_dayofweek"].astype(float) if "event_start_dayofweek" in d.columns else 0.0
    d["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    d["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)

    month = d["event_start_month"].astype(float) if "event_start_month" in d.columns else 1.0
    d["month_sin"] = np.sin(2 * np.pi * month / 12.0)
    d["month_cos"] = np.cos(2 * np.pi * month / 12.0)

    d = d.replace([np.inf, -np.inf], np.nan).fillna(0)

    return d



In [5]:
train_fe = build_features(train)
val_fe   = build_features(val)

drop_cols = ["event_id", "event", "time_to_hit_hours"]

X_tr = train_fe.drop(columns=drop_cols)
t_tr = train_fe["time_to_hit_hours"].values.astype(float)
e_tr = train_fe["event"].values.astype(int)

X_va = val_fe.drop(columns=drop_cols)
t_va = val_fe["time_to_hit_hours"].values.astype(float)
e_va = val_fe["event"].values.astype(int)

print("Train X shape:", X_tr.shape)
print("Val   X shape:", X_va.shape)

Train X shape: (198, 87)
Val   X shape: (23, 87)


In [6]:
def km_censor_survival(times, censor_event):
    order = np.argsort(times)
    times = times[order]
    censor_event = censor_event[order]

    uniq = np.unique(times)
    n = len(times)

    at_risk = n
    G = 1.0
    G_map = {}

    for tt in uniq:
        m = np.sum(times == tt)
        d = np.sum((times == tt) & (censor_event == 1))

        if at_risk > 0:
            G *= (1.0 - d / at_risk)

        G_map[tt] = max(G, 1e-6)
        at_risk -= m

    uniq_times = np.array(sorted(G_map.keys()))
    G_vals = np.array([G_map[u] for u in uniq_times])
    return uniq_times, G_vals


def G_of_t(t_query, uniq_times, G_vals):
    t_query = np.asarray(t_query)
    idx = np.searchsorted(uniq_times, t_query, side="right") - 1
    out = np.ones_like(t_query, dtype=float)
    ok = idx >= 0
    out[ok] = G_vals[idx[ok]]
    return np.clip(out, 1e-6, 1.0)

In [7]:
def enforce_monotone(P):
    P = np.clip(P, 0.0, 1.0)
    P[:, 1] = np.maximum(P[:, 1], P[:, 0])
    P[:, 2] = np.maximum(P[:, 2], P[:, 1])
    P[:, 3] = np.maximum(P[:, 3], P[:, 2])
    return np.clip(P, 0.0, 1.0)

def masked_brier_for_horizon(p, t, e, H):
    mask = []
    y = []
    for i in range(len(t)):
        if e[i] == 1 and t[i] <= H:
            mask.append(True); y.append(1)
        elif t[i] > H:
            mask.append(True); y.append(0)
        else:
            mask.append(False); y.append(0)
    mask = np.array(mask, dtype=bool)
    y = np.array(y, dtype=int)
    if mask.sum() == 0:
        return 0.0
    return np.mean((p[mask] - y[mask])**2)

In [8]:
def make_ipcw_data(X, t, e, H, uniq_times, G_vals, weight_cap=30.0):

    y = ((e == 1) & (t <= H)).astype(int)

    mask = ((e == 1) & (t <= H)) | (t > H)

    t_clip = np.minimum(t, H)

    G_t = G_of_t(t_clip, uniq_times, G_vals)

    w = 1.0 / G_t
    w = np.clip(w, 1.0, weight_cap)

    return X[mask], y[mask], w[mask], mask


def make_tail_data(X, t, e, H0, H1, uniq_times, G_vals, weight_cap=50.0):
    y = ((e == 1) & (t > H0) & (t <= H1)).astype(int)

    mask = ((e == 1) & (t > H0) & (t <= H1)) | (t > H1)

    t_clip = np.minimum(t, H1)
    G_t = G_of_t(t_clip, uniq_times, G_vals)
    w = 1.0 / G_t
    w = np.clip(w, 1.0, weight_cap)

    return X[mask], y[mask], w[mask], mask

In [9]:
def bagged_lgb_predict(X_tr, y_tr, w_tr, X_va, n_seeds=5):

    preds = np.zeros(len(X_va))

    for seed in range(42, 42 + n_seeds):
        model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.02,
            num_leaves=15,
            max_depth=3,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=8.0,
            objective="binary",
            random_state=seed,
            verbose=-1
        )

        model.fit(X_tr, y_tr, sample_weight=w_tr)
        preds += model.predict_proba(X_va)[:, 1]

    return preds / n_seeds

def bagged_xgb_predict(X_tr, y_tr, w_tr, X_va, n_seeds=5):
    preds = np.zeros(len(X_va))

    for seed in range(42, 42 + n_seeds):
        model = xgb.XGBClassifier(
                n_estimators=2500,
                learning_rate=0.03,
                max_depth=8,
                min_child_weight=1,
                subsample=0.9,
                colsample_bytree=0.6, 
                reg_alpha=0.5,
                reg_lambda=1.0,
                gamma=2.0,  
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=seed,
        )
        model.fit(X_tr, y_tr, sample_weight=w_tr)
        preds += model.predict_proba(X_va)[:, 1]

    return preds / n_seeds

def bagged_cb_predict(X_tr, y_tr, w_tr, X_va, n_seeds=5):
    preds = np.zeros(len(X_va))

    for seed in range(42, 42 + n_seeds):
        model = CatBoostClassifier(
            iterations=4000,
            learning_rate=0.03,
            depth=4,
            l2_leaf_reg=8.0,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=seed,
            verbose=False
        )
        model.fit(X_tr, y_tr, sample_weight=w_tr)
        preds += model.predict_proba(X_va)[:, 1]

    return preds / n_seeds

In [10]:
def run_pipeline(X_tr, t_tr, e_tr, X_va, t_va, e_va):

    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    H = 12
    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
        X_tr, t_tr, e_tr, H, uniq_times, G_vals
    )
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(
        X_va, t_va, e_va, H, uniq_times, G_vals
    )

    p12_full = np.zeros(len(X_va))
    p12_full[mask_va] = bagged_lgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=10)
    H = 24
    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
        X_tr, t_tr, e_tr, H, uniq_times, G_vals
    )
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(
        X_va, t_va, e_va, H, uniq_times, G_vals
    )

    p24_full = np.zeros(len(X_va))
    p24_full[mask_va] = bagged_lgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=10)

    X48_tr, y48_tr, w48_tr, m48_tr = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
    X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

    tail_feats = [
        "dist_km",
        "eta_effective_h",
        "effective_closing_speed",
        "alignment_abs",
        "has_movement"
    ]

    lr_48 = LogisticRegression(
        C=0.2,
        solver="lbfgs",
        max_iter=2000
    )

    lr_48.fit(
        X48_tr[tail_feats],
        y48_tr,
        sample_weight=w48_tr
    )

    p48_tail_full = np.zeros(len(X_va))
    p48_tail_full[m48_va] = lr_48.predict_proba(
        X48_va[tail_feats]
    )[:, 1]

    p48_full = p24_full + (1 - p24_full) * p48_tail_full

    late_events = np.sum((e_tr == 1) & (t_tr > 48))
    survive_48 = np.sum(t_tr > 48)
    alpha = (late_events + 1) / (survive_48 + 2)

    p72 = p48_full + (1 - p48_full) * alpha

    P = np.zeros((len(X_va), 4))
    P[:, 0] = p12_full
    P[:, 1] = p24_full
    P[:, 2] = p48_full
    P[:, 3] = p72

    P = enforce_monotone(P)

    b12 = masked_brier_for_horizon(P[:,0], t_va, e_va, 12)
    b24 = masked_brier_for_horizon(P[:,1], t_va, e_va, 24)
    b48 = masked_brier_for_horizon(P[:,2], t_va, e_va, 48)
    b72 = masked_brier_for_horizon(P[:,3], t_va, e_va, 72)

    print("Brier 12:", b12)
    print("Brier 24:", b24)
    print("Brier 48:", b48)
    print("Brier 72:", b72)

    return metric(P, t_va, e_va), P


def run_pipeline_lgb_wrapper(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    (c, b, s), P = run_pipeline(X_tr, t_tr, e_tr, X_va, t_va, e_va)
    return (c, b, s), P



def run_pipeline_xgb(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    H = 12
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)

    p12_full = np.zeros(len(X_va))
    p12_full[mask_va] = bagged_xgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=8)

    H = 24
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)

    p24_full = np.zeros(len(X_va))
    p24_full[mask_va] = bagged_xgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=8)

    X48_tr, y48_tr, w48_tr, _ = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
    X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

    tail_feats = [
        "dist_km",
        "eta_effective_h",
        "effective_closing_speed",
        "alignment_abs",
        "has_movement"
    ]

    p48_tail_full = np.zeros(len(X_va))
    p48_tail_full[m48_va] = bagged_xgb_predict(
        X48_tr[tail_feats], y48_tr, w48_tr,
        X48_va[tail_feats],
        n_seeds=10
    )

    p48_full = p24_full + (1 - p24_full) * p48_tail_full

    late_events = np.sum((e_tr == 1) & (t_tr > 48))
    survive_48 = np.sum(t_tr > 48)
    alpha = (late_events + 1) / (survive_48 + 2)

    p72 = p48_full + (1 - p48_full) * alpha
    
    P = np.zeros((len(X_va), 4))
    P[:, 0] = p12_full
    P[:, 1] = p24_full
    P[:, 2] = p48_full
    P[:, 3] = p72

    P = enforce_monotone(P)

    return metric(P, t_va, e_va), P

def run_pipeline_cat(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    H = 12
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)
    p12 = np.zeros(len(X_va)); p12[mask_va] = bagged_cb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=5)

    H = 24
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)
    p24 = np.zeros(len(X_va)); p24[mask_va] = bagged_cb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=5)

    X48_tr, y48_tr, w48_tr, _ = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
    X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

    tail_feats = ["dist_km","eta_effective_h","effective_closing_speed","alignment_abs","has_movement"]
    p48_tail = np.zeros(len(X_va))
    p48_tail[m48_va] = bagged_cb_predict(
        X48_tr[tail_feats], y48_tr, w48_tr,
        X48_va[tail_feats],
        n_seeds=5
    )
    p48 = p24 + (1 - p24) * p48_tail

    late_events = np.sum((e_tr == 1) & (t_tr > 48))
    survive_48 = np.sum(t_tr > 48)
    alpha = (late_events + 1) / (survive_48 + 2)

    p72 = p48 + (1 - p48) * alpha

    P = np.vstack([p12, p24, p48, p72]).T
    P = enforce_monotone(P)
    return metric(P, t_va, e_va), P

In [11]:
print("Training LGB...")
(_, _, _), P_lgb = run_pipeline_lgb_wrapper(X_tr, t_tr, e_tr, X_va, t_va, e_va)

print("Training XGB...")
(_, _, _), P_xgb = run_pipeline_xgb(X_tr, t_tr, e_tr, X_va, t_va, e_va)

print("Training CAT...")
(_, _, _), P_cat = run_pipeline_cat(X_tr, t_tr, e_tr, X_va, t_va, e_va)

print("\nVAL scores:")
print("LGB:", metric(P_lgb, t_va, e_va))
print("XGB:", metric(P_xgb, t_va, e_va))
print("CAT:", metric(P_cat, t_va, e_va))

Training LGB...


Brier 12: 0.04867129254182968
Brier 24: 0.03701005410567972
Brier 48: 0.05498293408629006
Brier 72: 0.04376598041216779
Training XGB...
Training CAT...

VAL scores:
LGB: (0.9367816091954023, np.float64(0.04622598398987028), np.float64(0.9486762939657114))
XGB: (0.9367816091954023, np.float64(0.03479206041111185), np.float64(0.9566800404708424))
CAT: (0.9310344827586207, np.float64(0.033078971996248915), np.float64(0.9561550644302119))


In [12]:
def fit_xgb_aft(X_tr, t_tr, e_tr, X_va, t_va, e_va, seed=42):
    lower_tr = np.log1p(t_tr.astype(float))
    upper_tr = np.log1p(t_tr.astype(float))
    upper_tr[e_tr == 0] = np.inf

    dtrain = xgb.DMatrix(X_tr)
    dtrain.set_float_info("label_lower_bound", lower_tr)
    dtrain.set_float_info("label_upper_bound", upper_tr)

    lower_va = np.log1p(t_va.astype(float))
    upper_va = np.log1p(t_va.astype(float))
    upper_va[e_va == 0] = np.inf

    dvalid = xgb.DMatrix(X_va)
    dvalid.set_float_info("label_lower_bound", lower_va)
    dvalid.set_float_info("label_upper_bound", upper_va)

    params = {
        "objective": "survival:aft",
        "aft_loss_distribution": "extreme",
        "aft_loss_distribution_scale": 1.5,
        "eta": 0.02,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.8,
        "lambda": 3.0,
        "alpha": 0.5,
        "tree_method": "hist",
        "seed": seed,
    }

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=5000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=200,
        verbose_eval=False
    )
    return model

def predict_aft_risk(model, X):
    d = xgb.DMatrix(X)
    pred_time = model.predict(d) 
    risk = -pred_time
    return risk

def run_pipeline_aft_risk(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    model = fit_xgb_aft(X_tr, t_tr, e_tr, X_va, t_va, e_va, seed=42)
    return predict_aft_risk(model, X_va)

In [13]:
def fit_xgb_cox(X_tr, t_tr, e_tr, X_va, t_va, e_va, seed=42):
    params = {
        "objective": "survival:cox",
        "eta": 0.02,
        "max_depth": 4,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "min_child_weight": 2,
        "lambda": 1.0,
        "alpha": 0.0,
        "seed": seed,
        "tree_method": "hist",
        "eval_metric": "cox-nloglik",
    }
    y_tr = t_tr.astype(float).copy()
    y_tr[e_tr == 0] *= -1.0

    y_va = t_va.astype(float).copy()
    y_va[e_va == 0] *= -1.0

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dvalid = xgb.DMatrix(X_va, label=y_va)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=5000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=200,
        verbose_eval=False
    )

    return model

def predict_cox_risk(model, X):
    d = xgb.DMatrix(X)
    risk = model.predict(d)  
    return risk

def run_pipeline_cox_risk(X_tr, t_tr, e_tr, X_va, t_va, e_va):

    model = fit_xgb_cox(
        X_tr, t_tr, e_tr,
        X_va, t_va, e_va,
        seed=42
    )

    risk_va = predict_cox_risk(model, X_va)
    return risk_va


In [14]:
def aft_probs_from_risk(model, X_tr, t_tr, e_tr, X_va, t_va, e_va):
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    risk_tr = predict_aft_risk(model, X_tr)
    risk_va = predict_aft_risk(model, X_va)

    P = np.zeros((len(X_va), 4))

    for j, H in enumerate([12,24,48,72]):

        Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
            X_tr, t_tr, e_tr, H, uniq_times, G_vals
        )

        r_tr = risk_tr[mask_tr]
        y_tr = yh_tr
        w_tr = wh_tr

        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(r_tr, y_tr, sample_weight=w_tr)

        _, _, _, mask_va = make_ipcw_data(
            X_va, t_va, e_va, H, uniq_times, G_vals
        )

        p = np.zeros(len(X_va))
        p[mask_va] = iso.predict(risk_va[mask_va])
        P[:, j] = p

    return enforce_monotone(P)


def run_pipeline_aft(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    model = fit_xgb_aft(
        X_tr, t_tr, e_tr,
        X_va, t_va, e_va,
        seed=42
    )
    P = aft_probs_from_risk(model, X_tr, t_tr, e_tr, X_va, t_va, e_va)
    return metric(P, t_va, e_va), P

In [15]:
def iso_map_risk_to_prob(risk_tr, X_tr, t_tr, e_tr, H, uniq_times, G_vals):
    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)

    r_tr = risk_tr[mask_tr]
    y_tr = yh_tr
    w_tr = wh_tr

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(r_tr, y_tr, sample_weight=w_tr)
    return iso, mask_tr

def cox_probs_from_risk(model, X_tr, t_tr, e_tr, X_va, t_va, e_va):
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    risk_tr = predict_cox_risk(model, X_tr)
    risk_va = predict_cox_risk(model, X_va)

    P = np.zeros((len(X_va), 4))

    for j, H in enumerate([12,24,48,72]):
        iso, mask_tr = iso_map_risk_to_prob(risk_tr, X_tr, t_tr, e_tr, H, uniq_times, G_vals)

        _, _, _, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)

        p = np.zeros(len(X_va))
        p[mask_va] = iso.predict(risk_va[mask_va])
        P[:, j] = p

    return enforce_monotone(P)

In [16]:
def risk_to_prob_matrix(risk, X, t, e):
    c_tr = (e == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t, c_tr)

    P = np.zeros((len(t), 4))

    for j, H in enumerate([12,24,48,72]):
        Xh, yh, wh, mask = make_ipcw_data(X, t, e, H, uniq_times, G_vals)

        r_tr = risk[mask]
        y_tr = yh
        w_tr = wh

        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(r_tr, y_tr, sample_weight=w_tr)

        p = np.zeros(len(t))
        p[mask] = iso.predict(risk[mask])
        P[:, j] = p

    return enforce_monotone(P)

In [17]:
def run_pipeline_cox(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    model = fit_xgb_cox(
        X_tr, t_tr, e_tr,
        X_va, t_va, e_va,
        seed=42
    )
    P = cox_probs_from_risk(model, X_tr, t_tr, e_tr, X_va, t_va, e_va)
    return metric(P, t_va, e_va), P

In [18]:
print("Training AFT model...")
model_aft = fit_xgb_aft(
    X_tr, t_tr, e_tr,
    X_va, t_va, e_va
)

print("Generating AFT risk...")
risk_tr = predict_aft_risk(model_aft, X_tr)
risk_va = predict_aft_risk(model_aft, X_va)

c_tr = (e_tr == 0).astype(int)
uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

P_risk_va = np.zeros((len(X_va), 4))

for j, H in enumerate([12,24,48,72]):

    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
        X_tr, t_tr, e_tr, H, uniq_times, G_vals
    )

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(risk_tr[mask_tr], yh_tr, sample_weight=wh_tr)

    _, _, _, mask_va = make_ipcw_data(
        X_va, t_va, e_va, H, uniq_times, G_vals
    )

    P_risk_va[mask_va, j] = iso.predict(risk_va[mask_va])

P_risk_va = enforce_monotone(P_risk_va)

print("AFT mapped score:", metric(P_risk_va, t_va, e_va))

Training AFT model...
Generating AFT risk...
AFT mapped score: (0.9195402298850575, np.float64(0.02731338247019224), np.float64(0.9567427012363826))


In [19]:
print("Training COX model...")
model_cox = fit_xgb_cox(
    X_tr, t_tr, e_tr,
    X_va, t_va, e_va,
    seed=42
)

print("Generating COX risk...")
risk_tr_cox = predict_cox_risk(model_cox, X_tr)
risk_va_cox = predict_cox_risk(model_cox, X_va)

P_cox_va = np.zeros((len(X_va), 4))

for j, H in enumerate([12,24,48,72]):

    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
        X_tr, t_tr, e_tr, H, uniq_times, G_vals
    )

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(risk_tr_cox[mask_tr], yh_tr, sample_weight=wh_tr)

    _, _, _, mask_va = make_ipcw_data(
        X_va, t_va, e_va, H, uniq_times, G_vals
    )

    P_cox_va[mask_va, j] = iso.predict(risk_va_cox[mask_va])

P_cox_va = enforce_monotone(P_cox_va)

print("COX mapped score:", metric(P_cox_va, t_va, e_va))

Training COX model...
Generating COX risk...
COX mapped score: (0.9080459770114943, np.float64(0.0196185060867239), np.float64(0.9586808388427415))


In [20]:
P_risk_mix = w_risk * P_cox_va + (1 - w_risk) * P_risk_va
P_risk_mix = enforce_monotone(P_risk_mix)

print("Risk mix score:", metric(P_risk_mix, t_va, e_va))

Risk mix score: (0.9195402298850575, np.float64(0.02731338247019224), np.float64(0.9567427012363826))


In [21]:
def blend_per_horizon(models, W):
    n = models[0].shape[0]
    P = np.zeros((n, 4))
    for j in range(4):
        for m_idx, Pi in enumerate(models):
            P[:, j] += W[j, m_idx] * Pi[:, j]
    return enforce_monotone(P)

models_va = [P_lgb, P_xgb, P_cat, P_risk_mix]

P_blend = blend_per_horizon(models_va, W)

def sharpen_p12(P, best_p):
    P = P.copy()
    P[:, 0] = np.power(P[:, 0], best_p)
    return enforce_monotone(P)
 
P_blend = sharpen_p12(P_blend, best_p)

print("Blended score BEFORE post-calibration:", metric(P_blend, t_va, e_va))

Blended score BEFORE post-calibration: (0.9310344827586207, np.float64(0.020359920979940695), np.float64(0.9650584001416277))
